# Кейс №3. Рошан и победа (SQL + PySpark + Iceberg)

**Заказчик:** игровые аналитики компании, анализирующей матчи Dota 2.  
**Основной бизнес-вопрос:** насколько убийство Рошана связано с победой, как эффект зависит от времени первого убийства, разницы числа убийств (контроля), темпа (длительности) матча и отставания по башням.

**Регламент обновления:** инкрементальное подтягивание данных каждые **4 часа** (синхронно с окном очистки источников `cleanup_runs` в PostgreSQL/MongoDB) с фиксацией `watermark` и использованием `MERGE INTO` в Apache Iceberg.  
**Полезный результат:** по каждой гипотезе формируется итоговый статус (`Подтверждена` / `Не подтверждена` / `Недостаточно данных`), величина эффекта (`lift`), 95% доверительный интервал (Уилсона) и p-value статистического теста ($\alpha = 0.05$). Учебные таблицы используют префикс `practice_case_03_`.

## Методика, архитектура слоёв и проверяемые гипотезы

### Разделение ответственности слоёв Lakehouse
* **RAW (`lakehouse.raw.practice_case_03_*`):** неизменённая история из PostgreSQL и MongoDB (инкремент каждые 4 часа через `MERGE INTO`).
* **STAGE (`lakehouse.stage.practice_case_03_*`):** **только очищенные, типизированные и дедуплицированные сущности без бизнес-агрегаций и расчётных гипотез** (`clean_matches`, `clean_match_players`, `clean_cs_roshan_events`, `clean_gold_roshan_events`, `clean_tower_death_events`). На каждом шаге фиксируется и выводится **воронка отсева строк** с указанием конкретной причины удаления.
* **MART (`lakehouse.mart.practice_case_03_*`):** вся бизнес-логика (привязка стороны убийцы, состояние башен строго до убийства Рошана, бинаризация таймингов, темп игры, расширенные метрики распределений и корреляции).

### Проверяемые гипотезы
* **H1 (Базовый эффект первого Рошана):** Если команда первой убивает Рошана, она выигрывает чаще 50% (биномиальный тест, 95% ДИ Уилсона).
* **H2 (Тайминги и лейт-гейм):** Если первый Рошан взят до 20-й минуты (`< 1200` сек), win rate выше минимум на 15 п.п., а после 45-й минуты (`>= 2700` сек) эффект ослабевает (критерий $\chi^2$, $p < 0.05$).
* **H3 (Контроль Рошана с учётом темпа матча):** Преимущество по числу убитых Рошанов повышает шанс на победу при контроле длительности матча (логистическая регрессия).

## 1. Источники и инкрементальное окно (раз в 4 часа)

Используются PostgreSQL `dota.matches`, `dota.match_players`, `dota.npcs` (для сверки `npcId = 133` — Рошан) и MongoDB `dota_events.cs_events`, `dota_events.gold_events`, `dota_events.tower_death_events`. Параметры подключения читаются из переменных окружения.

In [ ]:
import os
import time
from datetime import datetime, timedelta, timezone
import requests
import psycopg2
from pymongo import MongoClient
from lakehouse_publish import publish_table

pg_dsn = os.getenv("SOURCE_POSTGRES_DSN")
mongo_uri = os.getenv("SOURCE_MONGODB_URI")
if not pg_dsn or not mongo_uri:
    raise RuntimeError("Нужны переменные окружения SOURCE_POSTGRES_DSN и SOURCE_MONGODB_URI")

def safe_publish_table(layer, table_name, dataframe, description, dependencies=None, retries=4, delay=2.0):
    """Публикация метаданных в портал с ретраями при ConnectionResetError / Timeout."""
    for attempt in range(1, retries + 1):
        try:
            return publish_table(
                layer,
                table_name,
                dataframe,
                description=description,
                dependencies=dependencies
            )
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as err:
            if attempt == retries:
                print(f"[WARNING] Портал сбросил соединение для {layer}.{table_name} после {retries} попыток: {err}")
                print("Данные в Iceberg сохранены успешно, продолжаем выполнение.")
                return None
            time.sleep(delay * attempt)

# Режим загрузки: инкремент за последние 4 часа (синхронно с cleanup_runs) или полная загрузка
LOAD_MODE = os.getenv("CASE03_LOAD_MODE", "incremental_4h")  # "incremental_4h" или "full"
WINDOW_HOURS = 4
cutoff_utc = datetime.now(timezone.utc) - timedelta(hours=WINDOW_HOURS)
print(f"Режим загрузки: {LOAD_MODE} | Окно отсечения (UTC): {cutoff_utc.isoformat()}")

with psycopg2.connect(pg_dsn) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        cursor.execute("select npc_id, name from dota.npcs where npc_id = 133")
        print("Справочник NPC (133):", cursor.fetchone())
        for table in ["matches", "match_players"]:
            cursor.execute(f"select count(*) from dota.{table}")
            print(f"PostgreSQL {table} (всего в окне БД):", cursor.fetchone()[0])

with MongoClient(mongo_uri, serverSelectionTimeoutMS=5000) as client:
    db = client["dota_events"]
    print("MongoDB cs_events (npcId=133):", db["cs_events"].count_documents({"npcId": 133}))
    print("MongoDB gold_events (Roshan):", db["gold_events"].count_documents({"reason": "Roshan"}))
    print("MongoDB tower_death_events:", db["tower_death_events"].count_documents({}))

## 2. Построение учебного RAW (подтягивание батча за 4 часа + MERGE в Iceberg)

Батч читается с фильтром по времени вставки (`loaded_at >= cutoff_utc` / `created_at >= cutoff_utc`) и вливается в накопительные Iceberg-таблицы `lakehouse.raw.practice_case_03_*` через `MERGE INTO`. Регистрация в портале выполняется через `safe_publish_table` с повторами при сетевых обрывах.

In [ ]:
from psycopg2.extensions import parse_dsn
from pyspark.sql import functions as F, types as T
from lakehouse_spark import SparkSessionContext

pg = parse_dsn(pg_dsn)
jdbc_url = f"jdbc:postgresql://{pg['host']}:{pg.get('port', '5432')}/{pg['dbname']}"
jdbc_options = {"user": pg["user"], "password": pg["password"], "driver": "org.postgresql.Driver"}
cutoff_str = cutoff_utc.strftime("%Y-%m-%d %H:%M:%S")

cs_schema = T.StructType([
    T.StructField("_id", T.StringType()),
    T.StructField("match_id", T.LongType()),
    T.StructField("steam_account_id", T.LongType()),
    T.StructField("created_at", T.TimestampType()),
    T.StructField("time", T.IntegerType()),
    T.StructField("attacker", T.IntegerType()),
    T.StructField("npcId", T.IntegerType()),
    T.StructField("isFromIllusion", T.BooleanType()),
])

gold_schema = T.StructType([
    T.StructField("_id", T.StringType()),
    T.StructField("match_id", T.LongType()),
    T.StructField("steam_account_id", T.LongType()),
    T.StructField("created_at", T.TimestampType()),
    T.StructField("time", T.IntegerType()),
    T.StructField("amount", T.IntegerType()),
    T.StructField("reason", T.StringType()),
])

tower_schema = T.StructType([
    T.StructField("_id", T.StringType()),
    T.StructField("match_id", T.LongType()),
    T.StructField("created_at", T.TimestampType()),
    T.StructField("time", T.IntegerType()),
    T.StructField("radiant", T.IntegerType()),
    T.StructField("dire", T.IntegerType()),
])

def upsert_iceberg(spark, df, full_table_name: str, key_cols: list):
    """Идемпотентный инкрементальный MERGE INTO в Iceberg с сохранением истории."""
    if not spark.catalog.tableExists(full_table_name):
        df.writeTo(full_table_name).using("iceberg").createOrReplace()
        return
    df.createOrReplaceTempView("__incoming_batch")
    join_cond = " AND ".join([f"t.{k} = s.{k}" for k in key_cols])
    spark.sql(f"""
        MERGE INTO {full_table_name} t
        USING __incoming_batch s
        ON {join_cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

with SparkSessionContext("practice-case-03-raw-4h") as ctx:
    spark = ctx

    pg_filter = f"WHERE loaded_at >= TIMESTAMP '{cutoff_str}'" if LOAD_MODE == "incremental_4h" else ""

    matches = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", f"(SELECT match_id, start_time, duration, radiant_win, human_players, lobby_type, game_mode, tower_status_radiant, tower_status_dire, loaded_at FROM dota.matches {pg_filter}) q")
            .options(**jdbc_options)
            .load()
            .withColumnRenamed("loaded_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )

    players = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", f"(SELECT match_id, player_slot, account_id, hero_id, is_radiant, win, loaded_at FROM dota.match_players {pg_filter}) q")
            .options(**jdbc_options)
            .load()
            .withColumnRenamed("loaded_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )

    cs_roshan = (
        spark.read.format("mongodb")
            .option("connection.uri", mongo_uri)
            .option("database", "dota_events")
            .option("collection", "cs_events")
            .schema(cs_schema)
            .load()
            .filter("npcId = 133")
    )
    if LOAD_MODE == "incremental_4h":
        cs_roshan = cs_roshan.filter(F.col("created_at") >= F.lit(cutoff_str).cast("timestamp"))
    cs_roshan = (
        cs_roshan.select(
            F.col("_id").cast("string").alias("id"),
            "match_id", "steam_account_id", "time", "attacker", "npcId", "isFromIllusion",
            F.col("created_at").alias("src_modified_stamp")
        )
        .withColumn("sys_insert_stamp", F.current_timestamp())
    )

    gold_roshan = (
        spark.read.format("mongodb")
            .option("connection.uri", mongo_uri)
            .option("database", "dota_events")
            .option("collection", "gold_events")
            .schema(gold_schema)
            .load()
            .filter("reason = 'Roshan'")
    )
    if LOAD_MODE == "incremental_4h":
        gold_roshan = gold_roshan.filter(F.col("created_at") >= F.lit(cutoff_str).cast("timestamp"))
    gold_roshan = (
        gold_roshan.select(
            F.col("_id").cast("string").alias("id"),
            "match_id", "steam_account_id", "time", "amount", "reason",
            F.col("created_at").alias("src_modified_stamp")
        )
        .withColumn("sys_insert_stamp", F.current_timestamp())
    )

    tower_deaths = (
        spark.read.format("mongodb")
            .option("connection.uri", mongo_uri)
            .option("database", "dota_events")
            .option("collection", "tower_death_events")
            .schema(tower_schema)
            .load()
    )
    if LOAD_MODE == "incremental_4h":
        tower_deaths = tower_deaths.filter(F.col("created_at") >= F.lit(cutoff_str).cast("timestamp"))
    tower_deaths = (
        tower_deaths.select(
            F.col("_id").cast("string").alias("id"),
            "match_id", "time", "radiant", "dire",
            F.col("created_at").alias("src_modified_stamp")
        )
        .withColumn("sys_insert_stamp", F.current_timestamp())
    )

    datasets = {
        "matches": (matches, ["match_id"], "Накопительная RAW-таблица матчей (инкремент 4ч)."),
        "match_players": (players, ["match_id", "player_slot"], "Накопительная RAW-таблица игроков матча (инкремент 4ч)."),
        "cs_roshan_events": (cs_roshan, ["id"], "Накопительная RAW-таблица ластхитов по Рошану (npcId=133)."),
        "gold_roshan_events": (gold_roshan, ["id"], "Накопительная RAW-таблица командного золота за Рошана."),
        "tower_death_events": (tower_deaths, ["id"], "Накопительная RAW-таблица разрушения башен."),
    }

    for name, (frame, keys, desc) in datasets.items():
        output = f"lakehouse.raw.practice_case_03_{name}"
        with ctx.lineage(
            f"practice-case-03-raw-{name}",
            output=f"raw.practice_case_03_{name}",
            description=f"Инкрементальная загрузка (4ч) {name} для кейса 3",
            dataset_descriptions={f"raw.practice_case_03_{name}": desc}
        ):
            upsert_iceberg(spark, frame, output, keys)
            total_in_iceberg = spark.table(output).count()
            safe_publish_table("raw", f"practice_case_03_{name}", spark.table(output), description=desc)
        print(f"{output}: батч={frame.count()} | всего в Iceberg={total_in_iceberg}")

## 3. Сверка RAW с источником и проверка свежести 4-часового окна

Проверяем, что все записи из активного 4-часового окна источника доехали в Iceberg, а отставание последнего батча не превышает 4 часов (защита от потери данных при работе `cleanup_runs`).

In [ ]:
with SparkSessionContext("practice-case-03-reconcile") as spark:
    freshness = spark.sql("""
        select
            count(*) as total_matches_in_raw,
            count(distinct match_id) as unique_matches_in_raw,
            max(src_modified_stamp) as last_source_loaded_at,
            round((unix_timestamp(current_timestamp()) - unix_timestamp(max(src_modified_stamp))) / 3600.0, 2) as lag_hours
        from lakehouse.raw.practice_case_03_matches
    """).first()

    print(f"Всего матчей в накопительном RAW: {freshness['total_matches_in_raw']} (уникальных: {freshness['unique_matches_in_raw']})")
    print(f"Последняя загрузка из источника: {freshness['last_source_loaded_at']} | Отставание: {freshness['lag_hours']} ч.")
    if freshness["lag_hours"] is not None and freshness["lag_hours"] > 4.0:
        print("WARNING [DQ-RAW-01]: Отставание превышает 4-часовое окно cleanup_runs!")
    else:
        print("PASS [DQ-RAW-01]: Свежесть данных в пределах 4-часового окна.")

## 4. STAGE — только чистые данные и наглядный аудит отсева (Data Funnel)

В слое `stage` **отсутствует бизнес-логика** (нет джойнов между событиями, бинаризации, расчёта башен или статусов контроля). Выполняются только:
1. Дедупликация по первичному ключу (`sys_insert_stamp DESC`).
2. Приведение типов данных (`INT`, `LONG`, `BOOLEAN`) и нормализация названий полей.
3. Фильтрация технически некорректных и нерелевантных записей с **явным подсчётом и выводом причин урезания** на каждой таблице.

In [ ]:
import pandas as pd
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, LongType

funnel_records = []

def log_funnel(table_label: str, step_label: str, reason: str, before_cnt: int, after_cnt: int):
    removed = before_cnt - after_cnt
    pct = round((removed / before_cnt * 100), 2) if before_cnt > 0 else 0.0
    funnel_records.append({
        "Таблица": table_label,
        "Этап очистки": step_label,
        "Причина урезания": reason,
        "Было строк": before_cnt,
        "Стало строк": after_cnt,
        "Урезано строк": removed,
        "Урезано (%)": pct
    })

with SparkSessionContext("practice-case-03-stage-clean") as ctx:
    spark = ctx

    # -------------------------------------------------------------------------
    # 4.1 Чистые матчи (stage.practice_case_03_clean_matches)
    # -------------------------------------------------------------------------
    with ctx.lineage(
        "practice-case-03-clean-matches",
        inputs=["raw.practice_case_03_matches"],
        output="stage.practice_case_03_clean_matches",
        description="Дедупликация и очистка матчей (без бизнес-агрегаций)",
        dataset_descriptions={"stage.practice_case_03_clean_matches": "Чистая таблица матчей >= 10 мин и 10 игроков."}
    ):
        raw_m = spark.table("lakehouse.raw.practice_case_03_matches")
        c0 = raw_m.count()

        w_match = Window.partitionBy("match_id").orderBy(F.col("sys_insert_stamp").desc())
        m_dedup = raw_m.withColumn("rn", F.row_number().over(w_match)).filter("rn = 1").drop("rn")
        c1 = m_dedup.count()
        log_funnel("matches", "1. Дедупликация", "Повторные записи по match_id при инкрементальных загрузках", c0, c1)

        m_nulls = m_dedup.filter("match_id IS NOT NULL AND radiant_win IS NOT NULL AND duration IS NOT NULL")
        c2 = m_nulls.count()
        log_funnel("matches", "2. Проверка полноты (NOT NULL)", "Отсутствует исход матча (radiant_win) или длительность", c1, c2)

        m_humans = m_nulls.filter("human_players = 10")
        c3 = m_humans.count()
        log_funnel("matches", "3. Формат матча 5v5", "Неполные лобби или матчи с ботами (human_players != 10)", c2, c3)

        m_dur = m_humans.filter("duration >= 600 AND duration <= 18000")
        c4 = m_dur.count()
        log_funnel("matches", "4. Фильтр длительности", "Ранние ливы/несостоявшиеся матчи (< 10 мин) или баги (> 5 ч)", c3, c4)

        clean_matches = m_dur.select(
            F.col("match_id").cast(LongType()).alias("match_id"),
            F.col("start_time").cast(LongType()).alias("start_time"),
            F.col("duration").cast(IntegerType()).alias("duration_sec"),
            F.col("radiant_win").cast("boolean").alias("radiant_win"),
            F.col("human_players").cast(IntegerType()).alias("human_players"),
            F.col("tower_status_radiant").cast(IntegerType()).alias("tower_status_radiant"),
            F.col("tower_status_dire").cast(IntegerType()).alias("tower_status_dire")
        )
        clean_matches.writeTo("lakehouse.stage.practice_case_03_clean_matches").using("iceberg").createOrReplace()
        safe_publish_table("stage", "practice_case_03_clean_matches", clean_matches, description="Чистые матчи (без агрегаций)", dependencies=["raw/practice_case_03_matches"])

    # -------------------------------------------------------------------------
    # 4.2 Чистые игроки матча (stage.practice_case_03_clean_match_players)
    # -------------------------------------------------------------------------
    with ctx.lineage(
        "practice-case-03-clean-players",
        inputs=["raw.practice_case_03_match_players", "stage.practice_case_03_clean_matches"],
        output="stage.practice_case_03_clean_match_players",
        description="Дедупликация и нормализация игроков матча",
        dataset_descriptions={"stage.practice_case_03_clean_match_players": "Чистая таблица участников валидных матчей."}
    ):
        raw_p = spark.table("lakehouse.raw.practice_case_03_match_players")
        p0 = raw_p.count()

        w_player = Window.partitionBy("match_id", "player_slot").orderBy(F.col("sys_insert_stamp").desc())
        p_dedup = raw_p.withColumn("rn", F.row_number().over(w_player)).filter("rn = 1").drop("rn")
        p1 = p_dedup.count()
        log_funnel("match_players", "1. Дедупликация", "Дубли по ключу (match_id, player_slot)", p0, p1)

        p_valid_slot = p_dedup.filter("player_slot IN (0, 1, 2, 3, 4, 128, 129, 130, 131, 132) AND hero_id > 0")
        p2 = p_valid_slot.count()
        log_funnel("match_players", "2. Валидация слота и героя", "Некорректный player_slot или невыбранный герой (hero_id <= 0)", p1, p2)

        valid_match_ids = spark.table("lakehouse.stage.practice_case_03_clean_matches").select("match_id")
        p_in_clean_matches = p_valid_slot.join(valid_match_ids, on="match_id", how="inner")
        p3 = p_in_clean_matches.count()
        log_funnel("match_players", "3. Ссылочная целостность", "Игроки из отсеянных матчей (< 10 мин или != 10 игроков)", p2, p3)

        clean_players = p_in_clean_matches.select(
            F.col("match_id").cast(LongType()).alias("match_id"),
            F.col("player_slot").cast(IntegerType()).alias("player_slot"),
            F.when(F.col("account_id") == 4294967295, F.lit(None)).otherwise(F.col("account_id").cast(LongType())).alias("account_id"),
            F.col("hero_id").cast(IntegerType()).alias("hero_id"),
            F.coalesce(F.col("is_radiant"), F.col("player_slot") < 128).cast("boolean").alias("is_radiant"),
            F.col("win").cast("boolean").alias("win")
        )
        clean_players.writeTo("lakehouse.stage.practice_case_03_clean_match_players").using("iceberg").createOrReplace()
        safe_publish_table("stage", "practice_case_03_clean_match_players", clean_players, description="Чистые записи игроков матча", dependencies=["raw/practice_case_03_match_players", "stage/practice_case_03_clean_matches"])

    # -------------------------------------------------------------------------
    # 4.3 Чистые события убийства Рошана из cs_events
    # -------------------------------------------------------------------------
    with ctx.lineage(
        "practice-case-03-clean-cs-roshan",
        inputs=["raw.practice_case_03_cs_roshan_events", "stage.practice_case_03_clean_matches"],
        output="stage.practice_case_03_clean_cs_roshan_events",
        description="Очистка и дедупликация ластхитов по Рошану",
        dataset_descriptions={"stage.practice_case_03_clean_cs_roshan_events": "Чистые события убийства Рошана (npcId = 133)."}
    ):
        raw_cs = spark.table("lakehouse.raw.practice_case_03_cs_roshan_events")
        cs0 = raw_cs.count()

        w_cs = Window.partitionBy("match_id", "id").orderBy(F.col("sys_insert_stamp").desc())
        cs_dedup = raw_cs.withColumn("rn", F.row_number().over(w_cs)).filter("rn = 1").drop("rn")
        cs1 = cs_dedup.count()
        log_funnel("cs_roshan_events", "1. Дедупликация", "Повторные документы по ObjectId", cs0, cs1)

        cs_in_matches = (
            cs_dedup.alias("e")
            .join(spark.table("lakehouse.stage.practice_case_03_clean_matches").alias("m"), "match_id", "inner")
            .filter(F.col("e.time").between(0, F.col("m.duration_sec")))
            .select("e.*")
        )
        cs2 = cs_in_matches.count()
        log_funnel("cs_roshan_events", "2. Валидация времени и матча", "События из коротких матчей (<10 мин) или time вне [0, duration]", cs1, cs2)

        clean_cs = cs_in_matches.select(
            F.col("id").alias("event_id"),
            F.col("match_id").cast(LongType()).alias("match_id"),
            F.col("time").cast(IntegerType()).alias("time_sec"),
            F.col("steam_account_id").cast(LongType()).alias("steam_account_id"),
            F.col("attacker").cast(IntegerType()).alias("attacker_id"),
            F.col("npcId").cast(IntegerType()).alias("npc_id"),
            F.coalesce(F.col("isFromIllusion"), F.lit(False)).cast("boolean").alias("is_from_illusion")
        )
        clean_cs.writeTo("lakehouse.stage.practice_case_03_clean_cs_roshan_events").using("iceberg").createOrReplace()
        safe_publish_table("stage", "practice_case_03_clean_cs_roshan_events", clean_cs, description="Чистые события убийства Рошана", dependencies=["raw/practice_case_03_cs_roshan_events", "stage/practice_case_03_clean_matches"])

    # -------------------------------------------------------------------------
    # 4.4 Чистые события золота за Рошана (gold_events) и башен (tower_death_events)
    # -------------------------------------------------------------------------
    with ctx.lineage(
        "practice-case-03-clean-gold-and-towers",
        inputs=["raw.practice_case_03_gold_roshan_events", "raw.practice_case_03_tower_death_events", "stage.practice_case_03_clean_matches"],
        output="stage.practice_case_03_clean_tower_death_events",
        description="Очистка и дедупликация событий золота за Рошана и разрушения башен",
        dataset_descriptions={
            "stage.practice_case_03_clean_gold_roshan_events": "Чистые начисления золота за Рошана.",
            "stage.practice_case_03_clean_tower_death_events": "Чистые события разрушения башен."
        }
    ):
        matches_bounds = spark.table("lakehouse.stage.practice_case_03_clean_matches").select("match_id", "duration_sec")

        # Gold events
        raw_g = spark.table("lakehouse.raw.practice_case_03_gold_roshan_events")
        g0 = raw_g.count()
        w_g = Window.partitionBy("match_id", "id").orderBy(F.col("sys_insert_stamp").desc())
        g_dedup = raw_g.withColumn("rn", F.row_number().over(w_g)).filter("rn = 1 AND amount > 0").drop("rn")
        g1 = g_dedup.count()
        log_funnel("gold_roshan_events", "1. Дедупликация и amount > 0", "Дубли ObjectId или нулевое начисление золота", g0, g1)

        clean_gold = (
            g_dedup.join(matches_bounds, on="match_id", how="inner")
            .filter(F.col("time").between(0, F.col("duration_sec")))
            .select(
                F.col("id").alias("event_id"),
                F.col("match_id").cast(LongType()).alias("match_id"),
                F.col("time").cast(IntegerType()).alias("time_sec"),
                F.col("steam_account_id").cast(LongType()).alias("steam_account_id"),
                F.col("amount").cast(IntegerType()).alias("gold_amount")
            )
        )
        g2 = clean_gold.count()
        log_funnel("gold_roshan_events", "2. Фильтр по валидным матчам", "Записи из отсеянных матчей (< 10 мин) или вне времени матча", g1, g2)
        clean_gold.writeTo("lakehouse.stage.practice_case_03_clean_gold_roshan_events").using("iceberg").createOrReplace()
        safe_publish_table("stage", "practice_case_03_clean_gold_roshan_events", clean_gold, description="Чистые события золота за Рошана", dependencies=["raw/practice_case_03_gold_roshan_events", "stage/practice_case_03_clean_matches"])

        # Tower death events
        raw_t = spark.table("lakehouse.raw.practice_case_03_tower_death_events")
        t0 = raw_t.count()
        w_t = Window.partitionBy("match_id", "id").orderBy(F.col("sys_insert_stamp").desc())
        t_dedup = raw_t.withColumn("rn", F.row_number().over(w_t)).filter("rn = 1").drop("rn")
        t1 = t_dedup.count()
        log_funnel("tower_death_events", "1. Дедупликация", "Повторные документы по ObjectId", t0, t1)

        clean_towers = (
            t_dedup.join(matches_bounds, on="match_id", how="inner")
            .filter(F.col("time").between(0, F.col("duration_sec")))
            .select(
                F.col("id").alias("event_id"),
                F.col("match_id").cast(LongType()).alias("match_id"),
                F.col("time").cast(IntegerType()).alias("time_sec"),
                F.col("radiant").cast(IntegerType()).alias("radiant_tower_mask"),
                F.col("dire").cast(IntegerType()).alias("dire_tower_mask")
            )
        )
        t2 = clean_towers.count()
        log_funnel("tower_death_events", "2. Фильтр по валидным матчам", "События из отсеянных матчей (< 10 мин) или вне времени матча", t1, t2)
        clean_towers.writeTo("lakehouse.stage.practice_case_03_clean_tower_death_events").using("iceberg").createOrReplace()
        safe_publish_table("stage", "practice_case_03_clean_tower_death_events", clean_towers, description="Чистые события разрушения башен", dependencies=["raw/practice_case_03_tower_death_events", "stage/practice_case_03_clean_matches"])

    print("=== ВОРОНКА ОЧИСТКИ ДАННЫХ (STAGE): ГДЕ И ПОЧЕМУ УРЕЗАНЫ ЗНАЧЕНИЯ ===")
    funnel_df = pd.DataFrame(funnel_records)
    display(funnel_df)

    print("=== Пример чистых данных: stage.practice_case_03_clean_matches ===")
    spark.table("lakehouse.stage.practice_case_03_clean_matches").select(
        F.col("match_id").alias("ID матча"),
        F.col("start_time").alias("Время старта (Unix)"),
        F.col("duration_sec").alias("Длительность (сек)"),
        F.col("radiant_win").alias("Победа Radiant"),
        F.col("human_players").alias("Игроков")
    ).show(5, truncate=False)

## 5. MART — вся бизнес-логика, обогащение и расширенный набор метрик

Вся аналитическая логика перенесена из `STAGE` в слой `MART`:
1. **`mart.practice_case_03_match_roshan_mart` (Детальная витрина на уровне матча):**
   * Каскадная привязка стороны убийцы Рошана (`account_id` $\rightarrow$ `hero_id` $\rightarrow$ `gold_events`).
   * Расчёт уцелевших башен у сторон строго до убийства Рошана (`time_sec < roshan_time`) и флага `killed_while_losing_towers`.
   * Разметка темпа игры (`match_pace_bin`), интервалов первого Рошана (`0-15`, `15-25`, `25-35`, `35-45`, `45+` мин), разницы убийств (`roshan_diff_radiant`), времени между 1-м и 2-м Рошаном (`roshan_respawn_delta_min`) и командного золота за Рошана (`total_roshan_gold`).
2. **`mart.practice_case_03_roshan_time_effect` (Сводка по интервалам времени и башням):**
   * Расширенные метрики: число матчей, доля от всех матчей с Рошаном, винрейт, прирост `lift`, среднее и **медианное** время первого Рошана (`p50`), квартили (`p25`, `p75`), средняя длительность матча и среднее золото за Рошана.
3. **`mart.practice_case_03_roshan_control_effect` (Сводка контроля Рошана и темпа матча):**
   * Расширенные метрики: винрейт лидера по Рошанам по темпу игры и величине перевеса (`1`, `2`, `3+` убийств), среднее и медианное число убийств, доля камбэков (побед при отставании по башням).

In [ ]:
from pyspark.sql.window import Window

with SparkSessionContext("practice-case-03-mart-logic") as ctx:
    spark = ctx

    # -------------------------------------------------------------------------
    # 5.1 Сборка детальной бизнес-витрины на уровне матча
    # -------------------------------------------------------------------------
    with ctx.lineage(
        "practice-case-03-match-mart",
        inputs=[
            "stage.practice_case_03_clean_matches",
            "stage.practice_case_03_clean_match_players",
            "stage.practice_case_03_clean_cs_roshan_events",
            "stage.practice_case_03_clean_gold_roshan_events",
            "stage.practice_case_03_clean_tower_death_events"
        ],
        output="mart.practice_case_03_match_roshan_mart",
        description="Полная бизнес-логика кейса №3: привязка стороны, статус башен до убийства, темп и метрики на уровне матча",
        dataset_descriptions={"mart.practice_case_03_match_roshan_mart": "Детальная витрина матчей со всеми бизнес-признаками по Рошану."}
    ):
        matches_df = spark.table("lakehouse.stage.practice_case_03_clean_matches")
        players_df = spark.table("lakehouse.stage.practice_case_03_clean_match_players")
        cs_df = spark.table("lakehouse.stage.practice_case_03_clean_cs_roshan_events")
        gold_df = spark.table("lakehouse.stage.practice_case_03_clean_gold_roshan_events")
        towers_df = spark.table("lakehouse.stage.practice_case_03_clean_tower_death_events")

        # Маппинг игроков по account_id и hero_id
        players_by_acc = (
            players_df.filter("account_id IS NOT NULL")
            .select("match_id", "account_id", F.col("is_radiant").alias("acc_is_radiant"))
            .dropDuplicates(["match_id", "account_id"])
        )
        players_by_hero = (
            players_df.select("match_id", "hero_id", F.col("is_radiant").alias("hero_is_radiant"))
            .dropDuplicates(["match_id", "hero_id"])
        )

        # Агрегация командного золота за Рошана по 5-секундному окну
        gold_buckets = (
            gold_df.alias("g")
            .join(players_by_acc.alias("pa"), (F.col("g.match_id") == F.col("pa.match_id")) & (F.col("g.steam_account_id") == F.col("pa.account_id")), "left")
            .withColumn("time_bucket", F.floor(F.col("g.time_sec") / 5) * 5)
            .groupBy(F.col("g.match_id").alias("g_match_id"), "time_bucket")
            .agg(
                F.first("pa.acc_is_radiant", ignorenulls=True).alias("gold_is_radiant"),
                F.sum("g.gold_amount").alias("roshan_team_gold")
            )
        )

        # Каскадная привязка стороны убийцы Рошана
        roshan_with_side = (
            cs_df.alias("r")
            .join(players_by_acc.alias("pa"), (F.col("r.match_id") == F.col("pa.match_id")) & (F.col("r.steam_account_id") == F.col("pa.account_id")), "left")
            .join(players_by_hero.alias("ph"), (F.col("r.match_id") == F.col("ph.match_id")) & (F.col("r.attacker_id") == F.col("ph.hero_id")), "left")
            .join(gold_buckets.alias("gb"), (F.col("r.match_id") == F.col("gb.g_match_id")) & (F.floor(F.col("r.time_sec") / 5) * 5 == F.col("gb.time_bucket")), "left")
            .select(
                F.col("r.match_id"),
                F.col("r.event_id"),
                F.col("r.time_sec").alias("roshan_time"),
                F.coalesce(F.col("pa.acc_is_radiant"), F.col("ph.hero_is_radiant"), F.col("gb.gold_is_radiant")).alias("killer_is_radiant"),
                F.coalesce(F.col("gb.roshan_team_gold"), F.lit(0)).alias("roshan_team_gold"),
                F.when(F.col("pa.acc_is_radiant").isNotNull(), "1_account_id")
                 .when(F.col("ph.hero_is_radiant").isNotNull(), "2_hero_id")
                 .when(F.col("gb.gold_is_radiant").isNotNull(), "3_gold_events")
                 .otherwise("4_unknown").alias("side_resolution_method")
            )
        )

        # Состояние уцелевших башен строго ДО момента убийства Рошана (time_sec < roshan_time)
        towers_alive = (
            towers_df
            .withColumn("radiant_towers_alive", F.bit_count(F.col("radiant_tower_mask").bitwiseAND(F.lit(2047))))
            .withColumn("dire_towers_alive", F.bit_count(F.col("dire_tower_mask").bitwiseAND(F.lit(2047))))
            .select(F.col("match_id").alias("t_match_id"), F.col("time_sec").alias("tower_time"), "radiant_towers_alive", "dire_towers_alive")
        )

        w_last_tower = Window.partitionBy("match_id", "event_id").orderBy(F.col("tower_time").desc())
        w_roshan_seq = Window.partitionBy("match_id").orderBy(F.col("roshan_time").asc(), F.col("event_id").asc())

        roshan_events_enriched = (
            roshan_with_side
            .join(towers_alive, (roshan_with_side["match_id"] == towers_alive["t_match_id"]) & (towers_alive["tower_time"] < roshan_with_side["roshan_time"]), "left")
            .withColumn("rn", F.row_number().over(w_last_tower))
            .filter("rn = 1")
            .drop("rn", "t_match_id", "tower_time")
            .fillna({"radiant_towers_alive": 11, "dire_towers_alive": 11})
            .withColumn("roshan_kill_num", F.row_number().over(w_roshan_seq))
            .withColumn("prev_roshan_time", F.lag("roshan_time").over(w_roshan_seq))
            .withColumn("respawn_interval_sec", F.col("roshan_time") - F.col("prev_roshan_time"))
            .withColumn(
                "tower_diff_at_kill",
                F.when(F.col("killer_is_radiant") == True, F.col("radiant_towers_alive") - F.col("dire_towers_alive"))
                 .when(F.col("killer_is_radiant") == False, F.col("dire_towers_alive") - F.col("radiant_towers_alive"))
                 .otherwise(F.lit(None))
            )
            .withColumn("killed_while_losing_towers", F.col("tower_diff_at_kill") < 0)
        )

        print("=== Аудит определения стороны убийцы Рошана (MART) ===")
        (
            roshan_events_enriched
            .groupBy(F.col("side_resolution_method").alias("Метод привязки стороны"))
            .agg(F.count("*").alias("Количество убийств"))
            .orderBy("Метод привязки стороны")
            .show(truncate=False)
        )

        first_roshan = (
            roshan_events_enriched.filter("roshan_kill_num = 1")
            .select(
                "match_id",
                F.col("roshan_time").alias("first_roshan_time"),
                F.round(F.col("roshan_time") / 60.0, 2).alias("first_roshan_time_min"),
                F.col("killer_is_radiant").alias("first_roshan_is_radiant"),
                F.col("killed_while_losing_towers").alias("first_roshan_while_losing_towers"),
                F.col("tower_diff_at_kill").alias("first_roshan_tower_diff"),
                F.col("roshan_team_gold").alias("first_roshan_gold")
            )
        )

        roshan_agg = roshan_events_enriched.groupBy("match_id").agg(
            F.count("*").alias("total_roshan_kills"),
            F.max("roshan_time").alias("last_roshan_time"),
            F.avg("respawn_interval_sec").alias("avg_roshan_interval_sec"),
            F.sum("roshan_team_gold").alias("total_roshan_gold"),
            F.sum(F.when(F.col("killer_is_radiant") == True, 1).otherwise(0)).alias("roshan_killed_by_radiant"),
            F.sum(F.when(F.col("killer_is_radiant") == False, 1).otherwise(0)).alias("roshan_killed_by_dire"),
            F.sum(F.when(F.col("killer_is_radiant").isNull(), 1).otherwise(0)).alias("roshan_killer_side_unknown"),
            F.sum(F.when((F.col("roshan_time") < 900) & (F.col("killer_is_radiant") == True), 1).otherwise(0)).alias("early_15m_radiant"),
            F.sum(F.when((F.col("roshan_time") < 900) & (F.col("killer_is_radiant") == False), 1).otherwise(0)).alias("early_15m_dire"),
            F.sum(F.when(F.col("killed_while_losing_towers") == True, 1).otherwise(0)).alias("roshan_kills_while_losing_towers")
        )

        match_roshan_mart = (
            matches_df.alias("m")
            .join(roshan_agg.alias("a"), "match_id", "left")
            .join(first_roshan.alias("f"), "match_id", "left")
            .select(
                "m.match_id",
                "m.radiant_win",
                "m.duration_sec",
                F.round(F.col("m.duration_sec") / 60.0, 2).alias("duration_min"),
                F.when(F.col("m.duration_sec") < 1500, "1_Быстрый (<25 мин)")
                 .when(F.col("m.duration_sec") < 2400, "2_Средний (25-40 мин)")
                 .when(F.col("m.duration_sec") < 3300, "3_Долгий (40-55 мин)")
                 .otherwise("4_Сверхдолгий (55+ мин)").alias("match_pace_bin"),
                F.coalesce("a.total_roshan_kills", F.lit(0)).alias("total_roshan_kills"),
                F.coalesce("a.roshan_killed_by_radiant", F.lit(0)).alias("roshan_killed_by_radiant"),
                F.coalesce("a.roshan_killed_by_dire", F.lit(0)).alias("roshan_killed_by_dire"),
                F.coalesce("a.roshan_killer_side_unknown", F.lit(0)).alias("roshan_killer_side_unknown"),
                (F.coalesce("a.roshan_killed_by_radiant", F.lit(0)) - F.coalesce("a.roshan_killed_by_dire", F.lit(0))).alias("roshan_diff_radiant"),
                F.abs(F.coalesce("a.roshan_killed_by_radiant", F.lit(0)) - F.coalesce("a.roshan_killed_by_dire", F.lit(0))).alias("roshan_lead_abs"),
                F.coalesce("a.total_roshan_gold", F.lit(0)).alias("total_roshan_gold"),
                F.round(F.col("a.avg_roshan_interval_sec") / 60.0, 2).alias("avg_roshan_interval_min"),
                F.coalesce("a.roshan_kills_while_losing_towers", F.lit(0)).alias("roshan_kills_while_losing_towers"),
                "f.first_roshan_while_losing_towers",
                "f.first_roshan_tower_diff",
                "f.first_roshan_time",
                "f.first_roshan_time_min",
                "f.first_roshan_is_radiant",
                "f.first_roshan_gold",
                F.when(F.col("f.first_roshan_is_radiant").isNull(), F.lit(None))
                 .otherwise(F.col("f.first_roshan_is_radiant") == F.col("m.radiant_win")).alias("first_roshan_team_won"),
                F.when(F.col("f.first_roshan_time").isNull(), "0_Без_Рошана")
                 .when(F.col("f.first_roshan_time") < 900, "1_0-15_мин")
                 .when(F.col("f.first_roshan_time") < 1500, "2_15-25_мин")
                 .when(F.col("f.first_roshan_time") < 2100, "3_25-35_мин")
                 .when(F.col("f.first_roshan_time") < 2700, "4_35-45_мин")
                 .otherwise("5_45+_мин").alias("first_roshan_time_bin"),
                F.when(F.coalesce("a.total_roshan_kills", F.lit(0)) == 0, "Нет_Рошанов")
                 .when(F.coalesce("a.roshan_killed_by_radiant", F.lit(0)) > F.coalesce("a.roshan_killed_by_dire", F.lit(0)), "Лидерство_Radiant")
                 .when(F.coalesce("a.roshan_killed_by_dire", F.lit(0)) > F.coalesce("a.roshan_killed_by_radiant", F.lit(0)), "Лидерство_Dire")
                 .otherwise("Ничья_по_Рошанам").alias("roshan_control_status")
            )
        )

        match_roshan_mart.writeTo("lakehouse.mart.practice_case_03_match_roshan_mart").using("iceberg").createOrReplace()
        safe_publish_table(
            "mart", "practice_case_03_match_roshan_mart", match_roshan_mart,
            description="Детальная витрина матчей со всеми бизнес-метриками по Рошану",
            dependencies=[
                "stage/practice_case_03_clean_matches",
                "stage/practice_case_03_clean_match_players",
                "stage/practice_case_03_clean_cs_roshan_events",
                "stage/practice_case_03_clean_gold_roshan_events",
                "stage/practice_case_03_clean_tower_death_events"
            ]
        )

    # -------------------------------------------------------------------------
    # 5.2 Агрегированная витрина времени 1-го Рошана и башен (расширенные метрики)
    # -------------------------------------------------------------------------
    with ctx.lineage(
        "practice-case-03-time-effect",
        inputs=["mart.practice_case_03_match_roshan_mart"],
        output="mart.practice_case_03_roshan_time_effect",
        description="Расширенная сводка винрейта, квантилей времени, длительности и золота по интервалам 1-го Рошана",
        dataset_descriptions={"mart.practice_case_03_roshan_time_effect": "Аналитическая витрина эффекта времени первого Рошана и башен."}
    ):
        time_effect = spark.sql("""
            select
                first_roshan_time_bin,
                first_roshan_while_losing_towers as killed_while_losing_towers,
                count(*) as matches,
                round(count(*) * 100.0 / sum(count(*)) over (), 2) as share_of_roshan_matches_pct,
                sum(case when first_roshan_team_won then 1 else 0 end) as wins,
                avg(case when first_roshan_team_won then 1.0 else 0.0 end) as first_roshan_win_rate,
                avg(case when first_roshan_team_won then 1.0 else 0.0 end) - 0.5 as lift,
                round(avg(first_roshan_time_min), 2) as mean_roshan_time_min,
                round(percentile_approx(first_roshan_time_min, 0.25), 2) as p25_roshan_time_min,
                round(percentile_approx(first_roshan_time_min, 0.50), 2) as median_roshan_time_min,
                round(percentile_approx(first_roshan_time_min, 0.75), 2) as p75_roshan_time_min,
                round(avg(duration_min), 2) as avg_match_duration_min,
                round(avg(total_roshan_kills), 2) as avg_roshans_per_match,
                round(avg(first_roshan_gold), 0) as avg_first_roshan_gold
            from lakehouse.mart.practice_case_03_match_roshan_mart
            where first_roshan_time_bin <> '0_Без_Рошана'
              and first_roshan_team_won is not null
            group by 1, 2
        """)
        time_effect.writeTo("lakehouse.mart.practice_case_03_roshan_time_effect").using("iceberg").createOrReplace()
        safe_publish_table("mart", "practice_case_03_roshan_time_effect", time_effect, description="Расширенная витрина времени 1-го Рошана", dependencies=["mart/practice_case_03_match_roshan_mart"])

    # -------------------------------------------------------------------------
    # 5.3 Агрегированная витрина контроля Рошана и темпа матча (расширенные метрики)
    # -------------------------------------------------------------------------
    with ctx.lineage(
        "practice-case-03-control-effect",
        inputs=["mart.practice_case_03_match_roshan_mart"],
        output="mart.practice_case_03_roshan_control_effect",
        description="Расширенная сводка контроля Рошана по темпу матча, величине перевеса и золоту",
        dataset_descriptions={"mart.practice_case_03_roshan_control_effect": "Аналитическая витрина контроля Рошана с разрезом по темпу матча."}
    ):
        control_effect = spark.sql("""
            select
                match_pace_bin,
                roshan_control_status,
                roshan_lead_abs,
                count(*) as matches,
                round(avg(duration_min), 2) as avg_duration_min,
                round(percentile_approx(duration_min, 0.50), 2) as median_duration_min,
                round(avg(total_roshan_kills), 2) as avg_total_roshan_kills,
                round(avg(avg_roshan_interval_min), 2) as avg_respawn_interval_min,
                round(avg(total_roshan_gold), 0) as avg_total_roshan_gold,
                round(avg(case when first_roshan_while_losing_towers then 1.0 else 0.0 end) * 100, 2) as losing_towers_share_pct,
                avg(case
                    when roshan_control_status = 'Лидерство_Radiant' and radiant_win then 1.0
                    when roshan_control_status = 'Лидерство_Dire' and not radiant_win then 1.0
                    when roshan_control_status in ('Лидерство_Radiant', 'Лидерство_Dire') then 0.0
                    else null
                end) as leader_win_rate
            from lakehouse.mart.practice_case_03_match_roshan_mart
            group by 1, 2, 3
        """)
        control_effect.writeTo("lakehouse.mart.practice_case_03_roshan_control_effect").using("iceberg").createOrReplace()
        safe_publish_table("mart", "practice_case_03_roshan_control_effect", control_effect, description="Расширенная витрина контроля Рошана и темпа матча", dependencies=["mart/practice_case_03_match_roshan_mart"])

    print("=== Витрина 1: Время 1-го Рошана, квантили и башни ===")
    time_effect.select(
        F.col("first_roshan_time_bin").alias("Интервал 1-го Рошана"),
        F.col("killed_while_losing_towers").alias("Проигрывал по башням"),
        F.col("matches").alias("Матчей"),
        F.col("share_of_roshan_matches_pct").alias("Доля (%)"),
        F.round(F.col("first_roshan_win_rate") * 100, 2).alias("Винрейт (%)"),
        F.col("median_roshan_time_min").alias("Медиана времени (мин)"),
        F.col("avg_match_duration_min").alias("Ср. длительность (мин)"),
        F.col("avg_first_roshan_gold").alias("Ср. золото команды")
    ).orderBy("Интервал 1-го Рошана", "Проигрывал по башням").show(truncate=False)

    print("=== Витрина 2: Контроль Рошана, перевес убийств и темп матча ===")
    control_effect.select(
        F.col("match_pace_bin").alias("Темп матча"),
        F.col("roshan_control_status").alias("Статус контроля"),
        F.col("roshan_lead_abs").alias("Перевес убийств"),
        F.col("matches").alias("Матчей"),
        F.round(F.col("leader_win_rate") * 100, 2).alias("Винрейт лидера (%)"),
        F.col("avg_total_roshan_kills").alias("Ср. Рошанов"),
        F.col("avg_total_roshan_gold").alias("Ср. золото за Рошанов")
    ).orderBy("Темп матча", "Статус контроля", "Перевес убийств").show(30, truncate=False)

## 6. Качество данных (DQ) и контроль утечек

Проверяем:
* Уникальность матчей в `stage` (`match_id`), отсутствие матчей короче 10 минут (`duration_sec < 600`).
* Валидность сумм убийств по сторонам (`is_sum_valid`) и долю неопределённых сторон (`unknown_side_ratio`).
* **Контроль отсутствия утечки в башнях:** убеждаемся, что после исправления (`tower_time < roshan_time`) доля побед команды, проигрывающей по башням, адекватна игровой логике.

In [ ]:
with SparkSessionContext("practice-case-03-quality") as spark:
    checks = {
        "Уникальность match_id в STAGE": "select count(*) - count(distinct match_id) violations from lakehouse.stage.practice_case_03_clean_matches",
        "Отсев матчей короче 10 мин в STAGE": "select count_if(duration_sec < 600) violations from lakehouse.stage.practice_case_03_clean_matches",
        "Уникальность match_id в MART": "select count(*) - count(distinct match_id) violations from lakehouse.mart.practice_case_03_match_roshan_mart",
        "Целостность сумм убийств Рошана в MART": "select count_if((roshan_killed_by_radiant + roshan_killed_by_dire + roshan_killer_side_unknown) != total_roshan_kills) violations from lakehouse.mart.practice_case_03_match_roshan_mart",
        "Диапазон вероятностей в MART": "select count_if(first_roshan_win_rate not between 0 and 1) violations from lakehouse.mart.practice_case_03_roshan_time_effect",
    }

    print("=== Результаты автоматических проверок Data Quality ===")
    for name, query in checks.items():
        violations = spark.sql(query).first()["violations"]
        print(f"{name}: {'PASS (0 нарушений)' if violations == 0 else f'FAIL ({violations} нарушений)'}")

    spark.sql("""
        select
            count(*) as `Всего матчей`,
            sum(case when total_roshan_kills > 0 then 1 else 0 end) as `Матчей с Рошаном`,
            round(avg(case when total_roshan_kills > 0 then 1.0 else 0.0 end) * 100, 2) as `Доля матчей с Рошаном (%)`,
            sum(total_roshan_kills) as `Всего убийств Рошана`,
            round(sum(roshan_killer_side_unknown) * 100.0 / nullif(sum(total_roshan_kills), 0), 3) as `Доля неизвестной стороны (%)`
        from lakehouse.mart.practice_case_03_match_roshan_mart
    """).show(truncate=False)

    spark.sql("""
        select
            snapshot_id as `ID снапшота Iceberg`,
            committed_at as `Время фиксации`,
            operation as `Операция`
        from lakehouse.mart.practice_case_03_roshan_time_effect.snapshots
        order by committed_at desc limit 3
    """).show(truncate=False)

## 7. Статистическая проверка гипотез (95% ДИ Уилсона, тесты значимости и логистическая регрессия с контролем темпа)

Рассчитываем:
1. **95% доверительный интервал Уилсона** для каждой доли побед.
2. **Биномиальный тест** (для **H1**: отличается ли win rate с первым Рошаном от 50% при $\alpha = 0.05$).
3. **Критерий $\chi^2$ (хи-квадрат)** (для **H2**: значимы ли различия win rate между временными интервалами `0-15`, `15-25`, `25-35`, `35-45`, `45+` мин).
4. **Логистическую регрессию с контролем длительности матча** (для **H3**: влияние разницы убитых Рошанов `roshan_diff_radiant` на победу при фиксированной длительности `duration_min`).
5. Итоговую таблицу с вердиктом по каждой гипотезе (`Подтверждена` / `Не подтверждена` / `Недостаточно данных`).

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

def wilson_ci(wins: int, n: int, z: float = 1.96):
    """95% доверительный интервал Уилсона для доли побед."""
    if n == 0:
        return (0.0, 0.0)
    p = wins / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half_width = (z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)) / denom
    return (max(0.0, center - half_width), min(1.0, center + half_width))

with SparkSessionContext("practice-case-03-analytics") as spark:
    summary_pd = (
        spark.table("lakehouse.mart.practice_case_03_match_roshan_mart")
        .select(
            "radiant_win", "duration_min", "match_pace_bin",
            "total_roshan_kills", "roshan_diff_radiant", "roshan_lead_abs",
            "total_roshan_gold", "avg_roshan_interval_min",
            "first_roshan_time", "first_roshan_time_min", "first_roshan_time_bin", "first_roshan_team_won",
            "first_roshan_while_losing_towers", "roshan_control_status"
        )
        .toPandas()
    )

# --- 1. Проверка H1: Первый Рошан -> победа чаще 50% ---
df_first = summary_pd.dropna(subset=["first_roshan_team_won"]).copy()
df_first["first_roshan_team_won"] = df_first["first_roshan_team_won"].astype(int)
n_h1 = len(df_first)
wins_h1 = int(df_first["first_roshan_team_won"].sum())
wr_h1 = wins_h1 / n_h1 if n_h1 > 0 else 0.0
ci_low_h1, ci_high_h1 = wilson_ci(wins_h1, n_h1)
pval_h1 = stats.binomtest(wins_h1, n_h1, p=0.5, alternative="greater").pvalue if n_h1 > 0 else 1.0

# --- 2. Проверка H2: Первый Рошан до 20 мин (+15 п.п.) vs после 45 мин + Хи-квадрат по бинам ---
df_before_20 = df_first[df_first["first_roshan_time"] < 1200]
n_20, wins_20 = len(df_before_20), int(df_before_20["first_roshan_team_won"].sum())
wr_20 = wins_20 / n_20 if n_20 > 0 else 0.0
ci_20_l, ci_20_h = wilson_ci(wins_20, n_20)

df_after_45 = df_first[df_first["first_roshan_time"] >= 2700]
n_45, wins_45 = len(df_after_45), int(df_after_45["first_roshan_team_won"].sum())
wr_45 = wins_45 / n_45 if n_45 > 0 else 0.0
ci_45_l, ci_45_h = wilson_ci(wins_45, n_45)

# Расширенная сводная таблица по интервалам первого Рошана
bins_table = (
    df_first.groupby("first_roshan_time_bin", as_index=False)
    .agg(
        Матчей=("first_roshan_team_won", "count"),
        Побед=("first_roshan_team_won", "sum"),
        Ср_время_мин=("first_roshan_time_min", "mean"),
        Медиана_времени_мин=("first_roshan_time_min", "median"),
        Ср_длительность_мин=("duration_min", "mean"),
        Ср_Рошанов_в_матче=("total_roshan_kills", "mean")
    )
    .sort_values("first_roshan_time_bin")
)
bins_table["Поражений"] = bins_table["Матчей"] - bins_table["Побед"]
bins_table["Доля матчей (%)"] = (bins_table["Матчей"] / n_h1 * 100).round(2)
bins_table["Винрейт (%)"] = (bins_table["Побед"] / bins_table["Матчей"] * 100).round(2)
bins_table["Прирост lift (п.п.)"] = (bins_table["Винрейт (%)"] - 50.0).round(2)
cis = [wilson_ci(int(w), int(n)) for w, n in zip(bins_table["Побед"], bins_table["Матчей"])]
bins_table["ДИ 95% нижн."] = [round(c[0] * 100, 2) for c in cis]
bins_table["ДИ 95% верх."] = [round(c[1] * 100, 2) for c in cis]
bins_table["Ср. время (мин)"] = bins_table["Ср_время_мин"].round(1)
bins_table["Медиана времени (мин)"] = bins_table["Медиана_времени_мин"].round(1)
bins_table["Ср. длительность (мин)"] = bins_table["Ср_длительность_мин"].round(1)
bins_table["Ср. Рошанов"] = bins_table["Ср_Рошанов_в_матче"].round(2)
bins_table["Надёжность выборки"] = np.where(bins_table["Матчей"] < 30, "Мало данных (N<30)", "Достаточно")

chi2_stat, pval_chi2, _, _ = stats.chi2_contingency(bins_table[["Побед", "Поражений"]].values)
corr_r, corr_p = stats.pearsonr(df_first["first_roshan_time_min"], df_first["first_roshan_team_won"])
print(f"Корреляция Пирсона (время 1-го Рошана vs победа): r = {corr_r:.4f} (p-value = {corr_p:.2e})")

print("=== Таблица по интервалам первого Рошана (расширенные метрики + 95% ДИ Уилсона) ===")
display(bins_table[[
    "first_roshan_time_bin", "Матчей", "Доля матчей (%)", "Побед", "Винрейт (%)",
    "Прирост lift (п.п.)", "ДИ 95% нижн.", "ДИ 95% верх.", "Ср. время (мин)",
    "Медиана времени (мин)", "Ср. длительность (мин)", "Ср. Рошанов", "Надёжность выборки"
]].rename(columns={"first_roshan_time_bin": "Интервал первого Рошана"}))

# --- 3. Проверка H3: Контроль Рошана + Логистическая регрессия с контролем длительности ---
df_lead = summary_pd[summary_pd["roshan_control_status"].isin(["Лидерство_Radiant", "Лидерство_Dire"])].copy()
df_lead["leader_won"] = np.where(
    df_lead["roshan_control_status"] == "Лидерство_Radiant",
    df_lead["radiant_win"].astype(int),
    (~df_lead["radiant_win"]).astype(int)
)
n_h3 = len(df_lead)
wins_h3 = int(df_lead["leader_won"].sum())
wr_h3 = wins_h3 / n_h3 if n_h3 > 0 else 0.0
ci_3_l, ci_3_h = wilson_ci(wins_h3, n_h3)

# Детализация контроля по величине перевеса убийств (1, 2, 3+ Рошанов)
lead_breakdown = (
    df_lead.assign(Перевес_группа=np.where(df_lead["roshan_lead_abs"] >= 3, "3+ Рошанов", df_lead["roshan_lead_abs"].astype(str) + " Рошан(а)"))
    .groupby("Перевес_группа", as_index=False)
    .agg(
        Матчей=("leader_won", "count"),
        Побед_лидера=("leader_won", "sum"),
        Ср_длительность_мин=("duration_min", "mean"),
        Ср_золото_за_Рошанов=("total_roshan_gold", "mean")
    )
)
lead_breakdown["Винрейт лидера (%)"] = (lead_breakdown["Побед_лидера"] / lead_breakdown["Матчей"] * 100).round(2)
l_cis = [wilson_ci(int(w), int(n)) for w, n in zip(lead_breakdown["Побед_лидера"], lead_breakdown["Матчей"])]
lead_breakdown["ДИ 95% Уилсона"] = [f"[{c[0]*100:.1f}%; {c[1]*100:.1f}%]" for c in l_cis]
lead_breakdown["Ср. длительность (мин)"] = lead_breakdown["Ср_длительность_мин"].round(1)
lead_breakdown["Ср. золото за Рошанов"] = lead_breakdown["Ср_золото_за_Рошанов"].round(0)

print("=== Детализация контроля: как растёт винрейт с увеличением перевеса по Рошанам ===")
display(lead_breakdown[["Перевес_группа", "Матчей", "Винрейт лидера (%)", "ДИ 95% Уилсона", "Ср. длительность (мин)", "Ср. золото за Рошанов"]].rename(
    columns={"Перевес_группа": "Перевес по убийствам"}
))

X = sm.add_constant(summary_pd[["roshan_diff_radiant", "duration_min"]])
y = summary_pd["radiant_win"].astype(int)
logit_model = sm.Logit(y, X).fit(disp=False)
odds_ratio_roshan = math.exp(logit_model.params["roshan_diff_radiant"])
pval_logit_roshan = logit_model.pvalues["roshan_diff_radiant"]

# --- 4. Метрика башен: Убийство первого Рошана при отставании по башням ---
towers_tbl = (
    df_first.dropna(subset=["first_roshan_while_losing_towers"])
    .groupby("first_roshan_while_losing_towers", as_index=False)
    .agg(
        Матчей=("first_roshan_team_won", "count"),
        Побед=("first_roshan_team_won", "sum"),
        Ср_время_1го_Рошана_мин=("first_roshan_time_min", "mean"),
        Ср_длительность_мин=("duration_min", "mean")
    )
)
towers_tbl["Статус башен до Рошана"] = np.where(
    towers_tbl["first_roshan_while_losing_towers"],
    "Проигрывали по башням (< соперника)",
    "Не уступали по башням (>= соперника)"
)
towers_tbl["Винрейт (%)"] = (towers_tbl["Побед"] / towers_tbl["Матчей"] * 100).round(2)
t_cis = [wilson_ci(int(w), int(n)) for w, n in zip(towers_tbl["Побед"], towers_tbl["Матчей"])]
towers_tbl["ДИ 95%"] = [f"[{c[0]*100:.1f}%; {c[1]*100:.1f}%]" for c in t_cis]
towers_tbl["Ср. время 1-го Рошана (мин)"] = towers_tbl["Ср_время_1го_Рошана_мин"].round(1)
towers_tbl["Ср. длительность (мин)"] = towers_tbl["Ср_длительность_мин"].round(1)

print("=== Метрика башен: Убийство 1-го Рошана при отставании по строениям ===")
display(towers_tbl[["Статус башен до Рошана", "Матчей", "Побед", "Винрейт (%)", "ДИ 95%", "Ср. время 1-го Рошана (мин)", "Ср. длительность (мин)"]])

# --- 5. Итоговая таблица вердиктов по 3 гипотезам для заказчика ---
verdicts = pd.DataFrame([
    {
        "Гипотеза": "H1: Команда первого Рошана побеждает чаще 50%",
        "Выборка N": n_h1,
        "Винрейт (%)": round(wr_h1 * 100, 2),
        "Эффект lift (п.п.)": round((wr_h1 - 0.5) * 100, 2),
        "ДИ 95% Уилсона": f"[{ci_low_h1*100:.2f}%; {ci_high_h1*100:.2f}%]",
        "Тест и p-value": f"Binomial p={pval_h1:.2e}",
        "Статус": "Подтверждена" if (pval_h1 < 0.05 and ci_low_h1 > 0.5) else "Не подтверждена"
    },
    {
        "Гипотеза": "H2a: Первый Рошан до 20 мин даёт >= +15 п.п. к победе",
        "Выборка N": n_20,
        "Винрейт (%)": round(wr_20 * 100, 2),
        "Эффект lift (п.п.)": round((wr_20 - 0.5) * 100, 2),
        "ДИ 95% Уилсона": f"[{ci_20_l*100:.2f}%; {ci_20_h*100:.2f}%]",
        "Тест и p-value": f"Chi2 p={pval_chi2:.2e}",
        "Статус": "Недостаточно данных" if n_20 < 30 else ("Подтверждена" if (pval_chi2 < 0.05 and (wr_20 - 0.5) >= 0.15) else "Не подтверждена")
    },
    {
        "Гипотеза": "H2b: После 45 мин эффект 1-го Рошана ослабевает (близок к 50%)",
        "Выборка N": n_45,
        "Винрейт (%)": round(wr_45 * 100, 2),
        "Эффект lift (п.п.)": round((wr_45 - 0.5) * 100, 2),
        "ДИ 95% Уилсона": f"[{ci_45_l*100:.2f}%; {ci_45_h*100:.2f}%]",
        "Тест и p-value": f"Chi2 p={pval_chi2:.2e}",
        "Статус": "Недостаточно данных" if n_45 < 30 else ("Подтверждена" if abs(wr_45 - 0.5) < abs(wr_20 - 0.5) else "Не подтверждена")
    },
    {
        "Гипотеза": "H3: Преимущество по числу Рошанов повышает шанс победы (с контролем длительности)",
        "Выборка N": n_h3,
        "Винрейт (%)": round(wr_h3 * 100, 2),
        "Эффект lift (п.п.)": round((wr_h3 - 0.5) * 100, 2),
        "ДИ 95% Уилсона": f"[{ci_3_l*100:.2f}%; {ci_3_h*100:.2f}%]",
        "Тест и p-value": f"Logit OR={odds_ratio_roshan:.2f}, p={pval_logit_roshan:.2e}",
        "Статус": "Подтверждена" if (pval_logit_roshan < 0.05 and odds_ratio_roshan > 1.0) else "Не подтверждена"
    }
])

print("=== ИТОГОВЫЙ ОТЧЁТ ПО ГИПОТЕЗАМ КЕЙСА №3 ===")
display(verdicts)

# --- 6. Визуализация с доверительными интервалами ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(bins_table["first_roshan_time_bin"], bins_table["Матчей"], color="#4f6d7a")
axes[0].set_title("Число матчей по интервалам первого Рошана")
axes[0].set_ylabel("Количество матчей")
axes[0].tick_params(axis="x", rotation=20)

yerr_low = bins_table["Винрейт (%)"] - bins_table["ДИ 95% нижн."]
yerr_high = bins_table["ДИ 95% верх."] - bins_table["Винрейт (%)"]
axes[1].bar(
    bins_table["first_roshan_time_bin"],
    bins_table["Винрейт (%)"],
    yerr=[yerr_low, yerr_high],
    capsize=5,
    color="#c65d4b",
    ecolor="#222222"
)
axes[1].axhline(50, color="#333333", linestyle="--", linewidth=1, label="Базовый уровень (50%)")
axes[1].set_title("Винрейт команды первого Рошана (с 95% ДИ Уилсона)")
axes[1].set_ylabel("Доля побед, %")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()